# Predicting Electric Vehicle Purchases: Baseline Modeling

Kaggle Playground Series S6E9. Implements Phases 2–3 of
`docs/3_implementation_plan.md`: v1 sanity baselines, v2 strong models, and
the E01 budget-matched hand-designed configuration search, all on **fold
definition F1** with the promotion gate predeclared in
`docs/4_experiment_ledger.md`. Every run — including rejected ones — gets a
ledger row. The champion's fold-mean test predictions are written to
`submission.csv`.

EDA context (`docs/2_eda_insights.md`): top-heavy signal
(`Environmental_Concern_Level`, `Subsidy_Available`, `Annual_Income_USD`),
strictly monotone ordinals, one big interaction (the subsidy gate), no
missing values, no drift.

## 1. Config

In [1]:
import json
import platform
import resource
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lightgbm as lgb
from catboost import CatBoostClassifier

SEED = 42
N_SPLITS = 5  # fold definition F1 -- docs/4_experiment_ledger.md
TARGET = "Will_Buy_EV"
POSITIVE_CLASS = "Yes"
NOTEBOOK_VERSION = "v2"
BASELINE_CHAMPION = "v2b_catboost_default"  # ledger decision, 2026-09-01
N_BOOT = 1000  # paired stratified bootstrap draws (predeclared)

# Mode flags (master standard §4): flip off to skip expensive sections.
RUN_V1_SANITY = True
RUN_V2_STRONG = True
RUN_ANX_CATEGORICAL_AB = True
RUN_E01_TUNING = True
RUN_SUBMISSION = True

NUMERIC_FEATURES = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]
BASE_CATEGORICALS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
]
ANX = "Range_Anxiety_Level"
ANX_MAP = {"Low": 0, "Medium": 1, "High": 2}  # monotone -- EDA §4
ALL_FEATURES = NUMERIC_FEATURES + BASE_CATEGORICALS + [ANX]

print("python", platform.python_version())
print({m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)})

python 3.9.6
{'numpy': '2.0.2', 'pandas': '2.3.3', 'sklearn': '1.6.1', 'lightgbm': '4.6.0'}


## 2. Data Loading & Feature Frames

`Range_Anxiety_Level` is ordinal-encoded by default (strictly monotone
target rate — EDA §4); Section 6 A/Bs the categorical treatment. The other
five categoricals stay native (`category` dtype for HGB/LightGBM; named
`cat_features` for CatBoost).

In [2]:
def _find_data_dir() -> Path:
    """Locate the competition files on Kaggle or locally.

    Kaggle has mounted competition data at both
    /kaggle/input/competitions/<slug> and /kaggle/input/<slug> depending on
    the worker, so per master standard §12 the mount tree is walked rather
    than assumed when the known layouts miss.
    """
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e9"),
        Path("/kaggle/input/playground-series-s6e9"),
        Path("../data"),
        Path("data"),
    ]
    for cand in candidates:
        if (cand / "train.csv").exists():
            return cand
    mount = Path("/kaggle/input")
    if mount.exists():
        hits = sorted(mount.rglob("train.csv"))
        if hits:
            return hits[0].parent
    raise FileNotFoundError("train.csv not found in any known location")


DATA_DIR = _find_data_dir()
print(f"DATA_DIR = {DATA_DIR.resolve()}")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert train.shape == (668_665, 15) and test.shape == (286_571, 14)
assert train[ALL_FEATURES].isna().sum().sum() == 0
assert test[ALL_FEATURES].isna().sum().sum() == 0

y = (train[TARGET] == POSITIVE_CLASS).astype(int)


def make_features(
    df: pd.DataFrame, anx_as_categorical: bool = False
) -> pd.DataFrame:
    """Model-ready feature frame.

    Args:
        df: Raw train or test frame.
        anx_as_categorical: If True, keep Range_Anxiety_Level categorical
            instead of the default ordinal int encoding.

    Returns:
        Feature frame with category dtypes on the base categoricals.
    """
    frame = df[ALL_FEATURES].copy()
    if anx_as_categorical:
        frame[ANX] = frame[ANX].astype("category")
    else:
        frame[ANX] = frame[ANX].map(ANX_MAP).astype("int8")
    for col in BASE_CATEGORICALS:
        frame[col] = frame[col].astype("category")
    return frame


X = make_features(train)
X_test = make_features(test)
X_anxcat = make_features(train, anx_as_categorical=True)
X_test_anxcat = make_features(test, anx_as_categorical=True)
print(X.dtypes.to_string())

DATA_DIR = /Users/tuannm3812/Documents/GitHub/2. Kaggle/kaggle-s6e9-predicting-electric-vehicle-purchases/data


Age                               int64
Annual_Income_USD               float64
Daily_Commute_km                float64
Number_of_Cars_Owned              int64
Charging_Stations_Near_Home       int64
Charging_Stations_Near_Work       int64
Environmental_Concern_Level     float64
Gender                         category
City_Type                      category
Current_Car_Type               category
Home_Charging_Possible         category
Subsidy_Available              category
Range_Anxiety_Level                int8


## 3. Cross-Validation Harness (F1)

One harness for every model, so OOF predictions align row-for-row across
candidates (`docs/0_coding_standards.md`). Wall-clock and peak RSS are
recorded per run — the scale-override measurement.

In [3]:
results = []
oof_store = {}
test_store = {}


def peak_rss_gb() -> float:
    """Process peak RSS in GB (ru_maxrss is bytes on macOS, KB on Linux)."""
    raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return raw / (1024**3 if sys.platform == "darwin" else 1024**2)


def run_cv(name, model_factory, X_tr, X_te, cat_features=None):
    """5-fold OOF CV on F1; stores aligned OOF and fold-mean test preds.

    Args:
        name: Run name (becomes the ledger row key).
        model_factory: Zero-arg callable returning a fresh estimator.
        X_tr: Train features aligned with global `y`.
        X_te: Test features.
        cat_features: If set, passed to fit() (CatBoost path).

    Returns:
        (oof, test_pred) arrays.
    """
    skf = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=SEED
    )
    oof = np.zeros(len(X_tr))
    test_pred = np.zeros(len(X_te))
    fold_aucs = []
    t0 = time.time()
    for tr_idx, va_idx in skf.split(X_tr, y):
        model = model_factory()
        if cat_features is None:
            model.fit(X_tr.iloc[tr_idx], y.iloc[tr_idx])
        else:
            model.fit(
                X_tr.iloc[tr_idx], y.iloc[tr_idx],
                cat_features=cat_features,
            )
        oof[va_idx] = model.predict_proba(X_tr.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y.iloc[va_idx], oof[va_idx]))
        test_pred += model.predict_proba(X_te)[:, 1] / N_SPLITS
    elapsed = time.time() - t0
    row = {
        "run": name,
        "oof_auc": float(roc_auc_score(y, oof)),
        "fold_std": float(np.std(fold_aucs)),
        "fold_aucs": [round(float(a), 5) for a in fold_aucs],
        "wall_s": round(elapsed, 1),
        "peak_rss_gb": round(peak_rss_gb(), 2),
    }
    results.append(row)
    oof_store[name] = oof
    test_store[name] = test_pred
    print(
        f"{name:30s} OOF AUC {row['oof_auc']:.5f} ± {row['fold_std']:.5f} "
        f"| {row['wall_s']:7.1f}s | peak RSS {row['peak_rss_gb']:.2f} GB"
    )
    print(f"{'':30s} folds: {row['fold_aucs']}")
    return oof, test_pred

## 4. v1 — Sanity Baselines (+ First-Fit Measurement)

Constant predictor (floor), regularized logistic regression (linear floor —
sanity only, not a candidate: it cannot represent the subsidy gate without
an explicit product), and default `HistGradientBoostingClassifier`. The HGB
run doubles as the measured first full-data fit.

In [4]:
if RUN_V1_SANITY:
    const_auc = roc_auc_score(y, np.full(len(y), float(y.mean())))
    print(f"v1a_constant: AUC {const_auc:.3f} (rankless floor)")

v1a_constant: AUC 0.500 (rankless floor)


In [5]:
if RUN_V1_SANITY:

    def logistic_factory():
        pre = ColumnTransformer([
            ("num", StandardScaler(), NUMERIC_FEATURES + [ANX]),
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore"),
                BASE_CATEGORICALS,
            ),
        ])
        return Pipeline([
            ("pre", pre),
            ("clf", LogisticRegression(max_iter=2000)),
        ])

    run_cv("v1b_logistic", logistic_factory, X, X_test)

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


...

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ 

/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tuannm3812/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


v1b_logistic                   OOF AUC 0.93809 ± 0.00081 |     7.1s | peak RSS 0.75 GB
                               folds: [0.93667, 0.93803, 0.93907, 0.93862, 0.93807]


In [6]:
if RUN_V1_SANITY:
    run_cv(
        "v1c_hgb_default",
        lambda: HistGradientBoostingClassifier(
            random_state=SEED, categorical_features="from_dtype"
        ),
        X,
        X_test,
    )

v1c_hgb_default                OOF AUC 0.94102 ± 0.00087 |    17.9s | peak RSS 0.75 GB
                               folds: [0.9395, 0.94093, 0.94193, 0.94185, 0.94099]


**Insight:** the constant floor confirms the harness (AUC 0.500). The
logistic floor lands at **0.93809 ± 0.00081** — remarkably close to the
GBDTs (gap ≈ 0.003): despite the subsidy gate, most of the *ranking* is
recoverable additively in log-odds. Its solver emitted overflow
RuntimeWarnings (as in S6E8) — noted, not blocking: predictions are finite,
in-range, and sane. `v1c_hgb_default` reaches **0.94102 ± 0.00087** in
18.2 s at 0.99 GB peak RSS (local first run) — the measured first full-data
fit required by the scale override: full-data 5-fold CV is cheap here, and
no subsample regime is needed for standard experiments.

## 5. v2 — Strong Models (Native Categorical)

Untuned LightGBM and CatBoost with native categorical handling. Untuned
deliberately — the budget-matched comparison happens in Section 7 (E01),
not smuggled into the baseline.

In [7]:
if RUN_V2_STRONG:
    run_cv(
        "v2a_lightgbm_default",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X,
        X_test,
    )

v2a_lightgbm_default           OOF AUC 0.94115 ± 0.00082 |     8.5s | peak RSS 0.75 GB
                               folds: [0.93998, 0.94072, 0.94235, 0.94172, 0.94107]


In [8]:
if RUN_V2_STRONG:
    run_cv(
        "v2b_catboost_default",
        lambda: CatBoostClassifier(
            random_seed=SEED, verbose=0, allow_writing_files=False
        ),
        X,
        X_test,
        cat_features=BASE_CATEGORICALS,
    )

v2b_catboost_default           OOF AUC 0.94157 ± 0.00072 |   618.5s | peak RSS 1.19 GB
                               folds: [0.94045, 0.94121, 0.94248, 0.94221, 0.9415]


**Insight:** `v2a_lightgbm_default` **0.94115 ± 0.00082** (8.4 s local);
`v2b_catboost_default` **0.94157 ± 0.00072** (550.9 s local). All four GBDT
runs sit within 0.00055 of each other — the plateau EDA §3 predicted.
CatBoost's +0.00042 lead over LightGBM is smaller than the fold std
(~0.0007–0.0009), so this was recorded as a *working-champion* lead under
the predeclared highest-OOF rule, not a paired-gate promotion — and the v2
budgets are not comparable (1000 CatBoost iterations vs. 100 LightGBM
trees). Resolving that is exactly E01's job (Section 7).

## 6. `Range_Anxiety_Level` Representation A/B

Ordinal (default, monotone evidence) vs. native categorical, same family
and folds — the cheap A/B promised in the plan. Decided on OOF AUC delta
relative to fold std, not on a single-split number.

In [9]:
if RUN_ANX_CATEGORICAL_AB:
    run_cv(
        "v2c_lightgbm_anx_categorical",
        lambda: lgb.LGBMClassifier(random_state=SEED, verbose=-1),
        X_anxcat,
        X_test_anxcat,
    )

v2c_lightgbm_anx_categorical   OOF AUC 0.94123 ± 0.00088 |    10.7s | peak RSS 1.19 GB
                               folds: [0.9398, 0.94097, 0.94239, 0.94184, 0.94123]


**Insight:** categorical treatment scores 0.94123 vs. ordinal 0.94115 —
Δ = +0.00008, an order of magnitude below fold std. A tie: the ordinal
default stands (simpler, consistent with the monotone evidence), and the
A/B is logged in `docs/4_experiment_ledger.md` as resolved-no-effect, per
the plan's "logged either way" rule.

## 7. E01 — Budget-Matched Hand-Designed Configurations

The configuration list below was frozen in `docs/4_experiment_ledger.md`
**before** any E01 run; nothing is added or dropped afterwards. Comparable
boosting budgets across HGB / LightGBM / CatBoost — the three-family search
the v2 defaults could not provide.

In [10]:
E01_CONFIGS = [
    (
        "e01_hgb_1000x05",
        lambda: HistGradientBoostingClassifier(
            max_iter=1000, learning_rate=0.05, early_stopping=False,
            random_state=SEED, categorical_features="from_dtype",
        ),
    ),
    (
        "e01_hgb_2000x03_63l",
        lambda: HistGradientBoostingClassifier(
            max_iter=2000, learning_rate=0.03, max_leaf_nodes=63,
            early_stopping=False, random_state=SEED,
            categorical_features="from_dtype",
        ),
    ),
    (
        "e01_lgbm_1000x05",
        lambda: lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05,
            random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_lgbm_2000x03_63l",
        lambda: lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, num_leaves=63,
            min_child_samples=50, random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_lgbm_1000x05_127l",
        lambda: lgb.LGBMClassifier(
            n_estimators=1000, learning_rate=0.05, num_leaves=127,
            min_child_samples=100, colsample_bytree=0.8,
            random_state=SEED, verbose=-1,
        ),
    ),
    (
        "e01_cat_2000x05",
        lambda: CatBoostClassifier(
            iterations=2000, learning_rate=0.05, random_seed=SEED,
            verbose=0, allow_writing_files=False,
        ),
    ),
    (
        "e01_cat_1000x10_d8",
        lambda: CatBoostClassifier(
            iterations=1000, learning_rate=0.1, depth=8,
            random_seed=SEED, verbose=0, allow_writing_files=False,
        ),
    ),
]

if RUN_E01_TUNING:
    for name, factory in E01_CONFIGS:
        cats = BASE_CATEGORICALS if name.startswith("e01_cat") else None
        run_cv(name, factory, X, X_test, cat_features=cats)

e01_hgb_1000x05                OOF AUC 0.94145 ± 0.00069 |   280.0s | peak RSS 1.19 GB
                               folds: [0.94033, 0.94119, 0.94232, 0.94199, 0.94144]


e01_hgb_2000x03_63l            OOF AUC 0.94100 ± 0.00067 |   495.9s | peak RSS 1.19 GB
                               folds: [0.93999, 0.94073, 0.94203, 0.94132, 0.94095]


e01_lgbm_1000x05               OOF AUC 0.94155 ± 0.00079 |  2824.6s | peak RSS 1.19 GB
                               folds: [0.94028, 0.94128, 0.94267, 0.94203, 0.94156]


e01_lgbm_2000x03_63l           OOF AUC 0.94109 ± 0.00071 |   217.9s | peak RSS 1.19 GB
                               folds: [0.9401, 0.94078, 0.94223, 0.94143, 0.94097]


e01_lgbm_1000x05_127l          OOF AUC 0.94071 ± 0.00071 |   219.6s | peak RSS 1.19 GB
                               folds: [0.93962, 0.94048, 0.94175, 0.94113, 0.94062]


e01_cat_2000x05                OOF AUC 0.94176 ± 0.00074 |  1152.5s | peak RSS 1.23 GB
                               folds: [0.94059, 0.94143, 0.94272, 0.94234, 0.94175]


e01_cat_1000x10_d8             OOF AUC 0.94113 ± 0.00071 |  3978.6s | peak RSS 1.23 GB
                               folds: [0.9401, 0.94074, 0.94212, 0.94167, 0.94103]


**Insight:** the budget hypothesis held in *direction* but CatBoost kept
the crown: `e01_lgbm_1000x05` (0.94155) nearly closed the default-CatBoost
gap (0.94157) but did not pass it, while doubling CatBoost's budget
(`e01_cat_2000x05`, **0.94176**) topped the table. Every capacity increase
*hurt* — 127 leaves, depth 8, and the 2000×0.03 configs all scored below
their smaller siblings — so the plateau is regularization-side, not
capacity-starved. Wall-clocks in this pass are contention-inflated
(concurrent jobs shared the machine; e.g. 2824 s for the 1000-tree LightGBM
vs. 218 s for the 2000-tree one) — treat them as upper bounds, not
benchmarks.

## 8. Paired Promotion Gate

Predeclared in `docs/4_experiment_ledger.md`: a challenger is promoted over
`v2b_catboost_default` only if — on aligned F1 OOF predictions — it (1)
wins ≥ 3 of 5 folds, (2) the paired stratified bootstrap (B = 1000,
resampling within class, seed 42) 95% CI of ΔAUC is entirely positive, and
(3) P(Δ>0) ≥ 0.95. The bootstrap runs only for configs whose overall OOF
AUC exceeds the champion's point estimate; the rest are recorded as
not-promoted on the point estimate alone.

In [11]:
def paired_gate(cand: str, champ: str, n_boot: int = N_BOOT) -> dict:
    """Predeclared paired promotion gate on aligned F1 OOF predictions."""
    oof_c, oof_h = oof_store[cand], oof_store[champ]
    skf = StratifiedKFold(
        n_splits=N_SPLITS, shuffle=True, random_state=SEED
    )
    fold_deltas = []
    for _, va_idx in skf.split(X, y):
        y_va = y.iloc[va_idx]
        fold_deltas.append(
            roc_auc_score(y_va, oof_c[va_idx])
            - roc_auc_score(y_va, oof_h[va_idx])
        )
    wins = int(sum(d > 0 for d in fold_deltas))

    rng = np.random.RandomState(SEED)
    pos = np.where(y.values == 1)[0]
    neg = np.where(y.values == 0)[0]
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.r_[
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True),
        ]
        y_b = y.values[idx]
        deltas[b] = roc_auc_score(y_b, oof_c[idx]) - roc_auc_score(
            y_b, oof_h[idx]
        )
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p_pos = float((deltas > 0).mean())
    return {
        "fold_deltas": [round(float(d), 6) for d in fold_deltas],
        "fold_wins": wins,
        "ci95": (round(float(lo), 6), round(float(hi), 6)),
        "p_delta_pos": p_pos,
        "promoted": bool(wins >= 3 and lo > 0 and p_pos >= 0.95),
    }


gate_results = {}
if RUN_E01_TUNING:
    champ_auc = next(
        r["oof_auc"] for r in results if r["run"] == BASELINE_CHAMPION
    )
    challengers = [
        r["run"]
        for r in results
        if r["run"].startswith("e01_") and r["oof_auc"] > champ_auc
    ]
    print(
        f"champion {BASELINE_CHAMPION} OOF AUC {champ_auc:.5f}; "
        f"point-estimate challengers: {challengers or 'none'}"
    )
    for cand in challengers:
        gate_results[cand] = paired_gate(cand, BASELINE_CHAMPION)
        g = gate_results[cand]
        print(
            f"{cand:30s} fold wins {g['fold_wins']}/5 | "
            f"95% CI {g['ci95']} | P(d>0) {g['p_delta_pos']:.3f} "
            f"| promoted: {g['promoted']}"
        )

champion v2b_catboost_default OOF AUC 0.94157; point-estimate challengers: ['e01_cat_2000x05']


e01_cat_2000x05                fold wins 5/5 | 95% CI (0.000145, 0.000239) | P(d>0) 1.000 | promoted: True


**Insight:** exactly one config exceeded the champion's point estimate, so
one bootstrap ran. `e01_cat_2000x05` won **5/5 folds**, the paired 95% CI
(+0.000145, +0.000239) is entirely positive, and P(Δ>0) = 1.000 — all three
predeclared conditions hold, so it is **promoted**. The effect is real but
tiny (≈ +0.0002 AUC); expectations for the leaderboard delta should be
sized accordingly.

## 9. Summary, Sanity Checks, and Candidate Diversity

In [12]:
summary = (
    pd.DataFrame(results)
    .sort_values("oof_auc", ascending=False)
    .reset_index(drop=True)
)
summary

,run,oof_auc,fold_std,fold_aucs,wall_s,peak_rss_gb
0,e01_cat_2000x05,0.941759,0.000740,"[0.94059, 0.94143, 0.94272, 0.94234, 0.94175]",1152.5,1.23
1,v2b_catboost_default,0.941566,0.000723,"[0.94045, 0.94121, 0.94248, 0.94221, 0.9415]",618.5,1.19
2,e01_lgbm_1000x05,0.941554,0.000793,"[0.94028, 0.94128, 0.94267, 0.94203, 0.94156]",2824.6,1.19
3,e01_hgb_1000x05,0.941445,0.000689,"[0.94033, 0.94119, 0.94232, 0.94199, 0.94144]",280.0,1.19
4,v2c_lightgbm_anx_categorical,0.941230,0.000876,"[0.9398, 0.94097, 0.94239, 0.94184, 0.94123]",10.7,1.19
5,v2a_lightgbm_default,0.941150,0.000816,"[0.93998, 0.94072, 0.94235, 0.94172, 0.94107]",8.5,0.75
6,e01_cat_1000x10_d8,0.941127,0.000707,"[0.9401, 0.94074, 0.94212, 0.94167, 0.94103]",3978.6,1.23
7,e01_lgbm_2000x03_63l,0.941091,0.000707,"[0.9401, 0.94078, 0.94223, 0.94143, 0.94097]",217.9,1.19
8,v1c_hgb_default,0.941017,0.000875,"[0.9395, 0.94093, 0.94193, 0.94185, 0.94099]",17.9,0.75
9,e01_hgb_2000x03_63l,0.940995,0.000671,"[0.93999, 0.94073, 0.94203, 0.94132, 0.94095]",495.9,1.19


In [13]:
def candidate_sanity_checks(name: str) -> dict:
    """Finite, bounded, non-degenerate predictions; quantile comparison."""
    oof, test_pred = oof_store[name], test_store[name]
    return {
        "finite": bool(
            np.isfinite(oof).all() and np.isfinite(test_pred).all()
        ),
        "in_range": bool(
            (oof >= 0).all()
            and (oof <= 1).all()
            and (test_pred >= 0).all()
            and (test_pred <= 1).all()
        ),
        "oof_unique": int(np.unique(oof).size),
        "test_unique": int(np.unique(test_pred).size),
        "oof_q05_50_95": np.quantile(oof, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
        "test_q05_50_95": np.quantile(test_pred, [0.05, 0.5, 0.95])
        .round(4)
        .tolist(),
    }


for name in oof_store:
    print(name, candidate_sanity_checks(name))

oof_corr = pd.DataFrame(oof_store).corr().round(4)
print("\nOOF Pearson correlation (diversity bar: blend only if <= 0.995):")
print(oof_corr.to_string())

v1b_logistic {'finite': True, 'in_range': True, 'oof_unique': 668665, 'test_unique': 286571, 'oof_q05_50_95': [0.0001, 0.024, 0.7883], 'test_q05_50_95': [0.0001, 0.0242, 0.7891]}


v1c_hgb_default {'finite': True, 'in_range': True, 'oof_unique': 662582, 'test_unique': 286449, 'oof_q05_50_95': [0.0003, 0.0189, 0.7884], 'test_q05_50_95': [0.0003, 0.0194, 0.7879]}
v2a_lightgbm_default {'finite': True, 'in_range': True, 'oof_unique': 663453, 'test_unique': 286440, 'oof_q05_50_95': [0.0002, 0.0187, 0.7903], 'test_q05_50_95': [0.0002, 0.0191, 0.7895]}
v2b_catboost_default {'finite': True, 'in_range': True, 'oof_unique': 668654, 'test_unique': 286566, 'oof_q05_50_95': [0.0001, 0.0178, 0.8028], 'test_q05_50_95': [0.0001, 0.0183, 0.7999]}


v2c_lightgbm_anx_categorical {'finite': True, 'in_range': True, 'oof_unique': 664045, 'test_unique': 286408, 'oof_q05_50_95': [0.0003, 0.0188, 0.7904], 'test_q05_50_95': [0.0003, 0.0192, 0.7897]}


e01_hgb_1000x05 {'finite': True, 'in_range': True, 'oof_unique': 668067, 'test_unique': 286561, 'oof_q05_50_95': [0.0, 0.0179, 0.8], 'test_q05_50_95': [0.0, 0.0184, 0.7976]}
e01_hgb_2000x03_63l {'finite': True, 'in_range': True, 'oof_unique': 668636, 'test_unique': 286565, 'oof_q05_50_95': [0.0001, 0.0165, 0.8035], 'test_q05_50_95': [0.0001, 0.0172, 0.8005]}
e01_lgbm_1000x05 {'finite': True, 'in_range': True, 'oof_unique': 668118, 'test_unique': 286563, 'oof_q05_50_95': [0.0, 0.0178, 0.8006], 'test_q05_50_95': [0.0001, 0.0184, 0.7985]}


e01_lgbm_2000x03_63l {'finite': True, 'in_range': True, 'oof_unique': 668639, 'test_unique': 286566, 'oof_q05_50_95': [0.0001, 0.0168, 0.8054], 'test_q05_50_95': [0.0001, 0.0175, 0.8025]}


e01_lgbm_1000x05_127l {'finite': True, 'in_range': True, 'oof_unique': 668653, 'test_unique': 286566, 'oof_q05_50_95': [0.0, 0.0154, 0.8071], 'test_q05_50_95': [0.0, 0.0162, 0.8037]}
e01_cat_2000x05 {'finite': True, 'in_range': True, 'oof_unique': 668655, 'test_unique': 286567, 'oof_q05_50_95': [0.0001, 0.0182, 0.7997], 'test_q05_50_95': [0.0001, 0.0186, 0.7983]}
e01_cat_1000x10_d8 {'finite': True, 'in_range': True, 'oof_unique': 668655, 'test_unique': 286566, 'oof_q05_50_95': [0.0, 0.0173, 0.8032], 'test_q05_50_95': [0.0, 0.0179, 0.8001]}



OOF Pearson correlation (diversity bar: blend only if <= 0.995):
                              v1b_logistic  v1c_hgb_default  v2a_lightgbm_default  v2b_catboost_default  v2c_lightgbm_anx_categorical  e01_hgb_1000x05  e01_hgb_2000x03_63l  e01_lgbm_1000x05  e01_lgbm_2000x03_63l  e01_lgbm_1000x05_127l  e01_cat_2000x05  e01_cat_1000x10_d8
v1b_logistic                        1.0000           0.9887                0.9888                0.9815                        0.9890           0.9833               0.9805            0.9831                0.9791                 0.9774           0.9843              0.9808
v1c_hgb_default                     0.9887           1.0000                0.9979                0.9937                        0.9980           0.9956               0.9933            0.9954                0.9921                 0.9905           0.9957              0.9928
v2a_lightgbm_default                0.9888           0.9979                1.0000                0.9940               

**Insight:** twelve runs, the top eight within 0.0011 AUC — the plateau
EDA predicted, now measured. All candidates pass sanity (finite, bounded,
non-degenerate; train/test quantiles aligned). Diversity against the 0.995
bar, from the aligned OOF matrices: champion vs. `e01_lgbm_1000x05`
**0.9964** and vs. `e01_hgb_1000x05` **0.9963** — check per-pair: only pairs at or below 0.995 are blend-eligible
candidate, subject to the paired gate.

## 10. Champion & Prediction Artifacts

The champion is `v2b_catboost_default` unless a challenger cleared the
paired gate in Section 8, in which case the highest-OOF gate-cleared
challenger takes over. The logistic floor is excluded by predeclaration.
Aligned OOF/test matrices are persisted locally for later paired
comparisons and blend experiments.

In [14]:
promoted = [c for c, g in gate_results.items() if g["promoted"]]
if promoted:
    CHAMPION_NAME = max(
        promoted,
        key=lambda c: next(
            r["oof_auc"] for r in results if r["run"] == c
        ),
    )
    print(f"gate-promoted champion: {CHAMPION_NAME}")
else:
    CHAMPION_NAME = BASELINE_CHAMPION
    print(f"no challenger cleared the gate; champion stays {CHAMPION_NAME}")
champ_row = next(r for r in results if r["run"] == CHAMPION_NAME)
print(f"champion OOF AUC {champ_row['oof_auc']:.5f}")

# Local repo: ../predictions. On Kaggle: the kernel working dir, so
# `kaggle kernels output` can retrieve the matrices (docs/0 execution rule).
PRED_DIR = (
    Path("../predictions") if Path("../predictions").is_dir() else Path(".")
)
for name in oof_store:
    np.save(PRED_DIR / f"{name}_oof.npy", oof_store[name])
    np.save(PRED_DIR / f"{name}_test.npy", test_store[name])
print(f"aligned prediction matrices saved to {PRED_DIR.resolve()}")

gate-promoted champion: e01_cat_2000x05
champion OOF AUC 0.94176


aligned prediction matrices saved to /Users/tuannm3812/Documents/GitHub/2. Kaggle/kaggle-s6e9-predicting-electric-vehicle-purchases/predictions


## 11. Submission

In [15]:
if RUN_SUBMISSION:
    submission = pd.DataFrame(
        {"id": test["id"], TARGET: test_store[CHAMPION_NAME]}
    )
    assert submission.shape == sample_submission.shape
    assert (
        submission["id"].values == sample_submission["id"].values
    ).all()
    submission.to_csv("submission.csv", index=False)
    print(
        f"submission.csv written from {CHAMPION_NAME} "
        f"(notebook {NOTEBOOK_VERSION}); range "
        f"[{submission[TARGET].min():.4f}, "
        f"{submission[TARGET].max():.4f}]"
    )

submission.csv written from e01_cat_2000x05 (notebook v2); range [0.0000, 0.9893]


## 12. Next Moves

1. **Champion: `e01_cat_2000x05`** (OOF AUC 0.94176 ± 0.00074), promoted by
   the predeclared paired gate (5/5 folds, CI entirely positive) — recorded
   in `docs/4_experiment_ledger.md`.
2. **Submission:** this kernel's completed version carries the champion's
   fold-mean `submission.csv`; submit via `kaggle competitions submit -k
   tuannm3812/ev-purchases-baseline-modeling -v <version> -f
   submission.csv` and log it in `docs/5_submission_manifest.md`.
3. **Optuna: skipped**, per the plan's predeclared condition — seven
   hand-designed configs span only 0.94071–0.94176 with every capacity
   increase scoring worse; there is no evidenced headroom for automated
   search to exploit.
4. **Blend: skipped by the predeclared diversity bar** — champion OOF
   correlation is 0.9964 (LightGBM) / 0.9963 (HGB), both above 0.995. Same
   outcome as S6E8's 0.9976 skip: recorded, not attempted.
5. **Execution rule:** future runs execute on Kaggle
   (`docs/0_coding_standards.md`), so E02 lands in this kernel's next
   version, not a local run.

## Reproducibility Snapshot

In [16]:
snapshot = {
    "generated_utc": datetime.now(timezone.utc).isoformat(
        timespec="seconds"
    ),
    "notebook_version": NOTEBOOK_VERSION,
    "seed": SEED,
    "fold_definition": (
        f"F1: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, "
        f"random_state={SEED})"
    ),
    "champion": CHAMPION_NAME,
    "gate_results": gate_results,
    "results": results,
    "versions": {
        m.__name__: m.__version__ for m in (np, pd, sklearn, lgb)
    },
    "python": platform.python_version(),
}
print(json.dumps(snapshot, indent=2))

{
  "generated_utc": "2026-09-01T11:36:24+00:00",
  "notebook_version": "v2",
  "seed": 42,
  "fold_definition": "F1: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
  "champion": "e01_cat_2000x05",
  "gate_results": {
    "e01_cat_2000x05": {
      "fold_deltas": [
        0.000134,
        0.000224,
        0.000238,
        0.000133,
        0.000243
      ],
      "fold_wins": 5,
      "ci95": [
        0.000145,
        0.000239
      ],
      "p_delta_pos": 1.0,
      "promoted": true
    }
  },
  "results": [
    {
      "run": "v1b_logistic",
      "oof_auc": 0.9380892677192406,
      "fold_std": 0.0008073157075398691,
      "fold_aucs": [
        0.93667,
        0.93803,
        0.93907,
        0.93862,
        0.93807
      ],
      "wall_s": 7.1,
      "peak_rss_gb": 0.75
    },
    {
      "run": "v1c_hgb_default",
      "oof_auc": 0.9410172617838523,
      "fold_std": 0.000874832760267404,
      "fold_aucs": [
        0.9395,
        0.94093,
        0.94193